# Running the membrane thickness estimation pipeline

---

## Table of contents

1. [Introduction](#introduction)
2. [Changes from the previous pipeline version](#changes-from-the-previous-pipeline-version)
3. [Input data](#input-data)
4. [Setup](#setup)
5. [Parameters](#parameters)
6. [Run full pipeline](#run-full-pipeline)
7. [Run step 1: surface extraction](#run-step-1-surface-extraction)
8. [Run step 2: surface point matching](#run-step-2-surface-point-matching)
9. [Run step 3: intensity profile analysis](#run-step-3-intensity-profile-analysis)

**Companion tutorials:**  
[`memthick_analysis.ipynb`](memthick_analysis.ipynb) — loading results, plotting thickness distributions, 3D spatial maps, intensity profile summaries, and motl export.  
[`memthick_tomo_preprocessing.ipynb`](memthick_tomo_preprocessing.ipynb) — how the choice of tomogram reconstruction and denoising (WBP, Wiener-like filter deconvolution, cryoCARE) affects the measured membrane thickness.

---

## Introduction

This tutorial covers the `cryocat.analysis.memthick` pipeline for measuring membrane thickness from cryo-electron tomography (cryo-ET) membrane segmentations as described in [1].

The pipeline runs in three sequential steps:

| Step | What it does | Output |
|-------|-------------|--------|
| **1 — Surface extraction** | Runs marching cubes on each membrane label; refines surface normals; separates the two segmentation surfaces into surface 1 and surface 2 | `*_vertices_normals.csv` |
| **2 — Geometric matching** | Pairs each surface-1 vertex with its nearest surface-2 neighbor within a given distance and angle deviation from the vertex normal; records the Euclidean distance between paired points (`match_distance_nm`) | `*_matched_points.csv` |
| **3 — Intensity profile analysis** | Extracts short 1-D intensity profiles from the tomogram along each matched point pair; detects bilayer minima and outward maxima; computes inflection-point-based membrane thickness | `*_thickness.csv`, `*_int_profiles.pkl` |

`run_full_pipeline` runs all three steps in one call. Each step can also be called independently, allowing you to re-enter the pipeline at any point.

The input data used throughout this tutorial is a *Chlamydomonas reinhardtii* tomogram [2], with membrane segmentations generated using [MemBrain-seg](https://github.com/teamtomo/membrain-seg/tree/main) [3].

---

## Changes from the previous pipeline version

The previous version of the pipeline used tomogram intensity profiles as a **validation step**: thickness was first measured geometrically between matched segmentation surface points, and intensity profiles were then used to filter out unreliable measurements by checking whether both bilayer minima and a central maximum were detectable.

The current implementation uses the **tomogram intensity profiles as the primary source of the membrane thickness measurement**.

- Steps 1 and 2 (surface extraction and geometric matching) are unchanged. The matched-point distance (`match_distance_nm`) is still recorded and reflects the segmentation geometry.
- In step 3, intensity profiles are extracted along each matched pair. The pipeline then identifies the two leaflet **minima** (phosphate headgroups) and searches outward from each for an intensity **maximum** (where the density returns to baseline). The **inflection point** on the steepest slope between each minimum and its outward maximum defines the membrane boundary, and the distance between the two inflection points is the reported `membrane_thickness_nm`.
- This means `membrane_thickness_nm` is measured from the tomogram densities. The segmentation surface points are used only to anchor the profile extraction and guide the minima search windows.

For a detailed description of the detection logic, output columns, and `detection_mode` values (`max_max`, `max_anchor`, `minima_only`), see Section 1 of [`memthick_analysis.ipynb`](memthick_analysis.ipynb).

---

## Input data

- **Segmentation MRC** — instance segmentation with integer labels, one per membrane type. We used [MemBrain-seg](https://github.com/teamtomo/membrain-seg/tree/main) [3] for automated membrane segmentation.
- **Tomogram MRC** — the reconstruction used for intensity profile analysis (step 3). Must be on the same grid (same dimensions and voxel size) as the segmentation. See [`memthick_tomo_preprocessing.ipynb`](memthick_tomo_preprocessing.ipynb) for how the choice of reconstruction method affects measured thickness.

The example data used in this tutorial can be found [here](zenodolink).

#### References

[1] D. Glushkova, *et al.* Measuring membrane thickness in cryo-electron tomograms. *J. Cell Biol.* **225**, e202504053 (2026). [https://rupress.org/jcb/article/225/1/e202504053/278443](https://rupress.org/jcb/article/225/1/e202504053/278443)

[2] R. Kelley, *et al.* Towards community-driven visual proteomics with large-scale cryo-electron tomography of *Chlamydomonas reinhardtii*. *Mol. Cell* (2025). [https://doi.org/10.1016/j.molcel.2025.04.022](https://www.sciencedirect.com/science/article/pii/S1097276525009700?via%3Dihub)

[3] L. Lamm, *et al.* MemBrain v2: an end-to-end tool for the analysis of membranes in cryo-electron tomography. *bioRxiv* (2025). [https://www.biorxiv.org/content/10.1101/2024.01.05.574336v2](https://www.biorxiv.org/content/10.1101/2024.01.05.574336v2)


---

## Setup

Set paths to the input segmentation and tomogram MRC files and to the output directory. All pipeline outputs (CSVs, optional MRC volumes) will be written under `output_path`.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from cryocat.analysis import memthick

In [3]:
segmentation_path = "inputs/2140_z150to400_segmented.mrc"
tomo_path         = "inputs/2140_z150to400.mrc"
output_path        = "outputs"
os.makedirs(output_path, exist_ok=True)

---
## Parameters


#### Step 1: Surface extraction

| Parameter | Default | Description |
|-----------|---------|-------------|
| `smooth_sigma_segmentation` | `None` | Gaussian sigma (voxels) applied to the segmentation before running marching cubes. Smoothing blurs the binary mask into a soft field. `None` skips smoothing. Typical values: `0.5`–`2.5`. |
| `step_size_marching_cubes` | `1` | Marching-cubes stride. `1` produces the densest mesh (one vertex per voxel surface) |
| `surface_separation_mode` | `"planar"` | How the two bilayer leaflets are separated — see table below. Pass a `dict` (e.g. `{"ER": "planar", "OMM": "closed"}`) to use different modes per membrane. |
| `refine_normals` | `True` | Run neighbor-based normal averaging within each separated surface after the global orientation step. Strongly recommended: it minimizes systematic errors that the MST-based global orientation leaves on curved membranes. |
| `radius_hit` | `3.0` | Voxel-radius neighborhood used for normal refinement. Larger values average over a wider area (smoother normals, less sensitive to local mesh noise). |
| `snap_vertices_to_boundary` | `False` | Round marching-cubes vertices to integer voxels and discard any that are not on the segmentation boundary. Use when you want to run step 2 on confirmed boundary voxels only, e.g. for a segmentation-geometry-only analysis without intensity profiles. **Do not use when extracting intensity profiles** from the tomogram — sub-voxel vertex positions give better-centred profile sampling. |
| `save_split_surface_meshes` | `False` | Save each bilayer leaflet as a `.ply` mesh file. Useful for visualising the surface split (MeshLab, ParaView, Plotly). |
| `save_vertices_mrc` | `False` | Save vertex positions as a binary MRC volume (voxel = 1 at each vertex position). Useful for checking whether vertices cover the segmentation surface. |

#### `surface_separation_mode` — choosing the right mode

| Mode | When to use | How it works |
|------|-------------|--------------|
| `"planar"` | Flat or slightly curved membranes: nuclear envelope, ER, plasma membrane patches | PCA on all vertex coordinates. Vertices are split by which side of the membrane's principal plane they fall on. Fast and robust for planar geometry; fails for closed surfaces where both leaflets span the same plane. |
| `"closed"` | Closed organelle membranes: mitochondria (especially OMM), vesicles, lysosomes | Connected-component labeling on the mesh. The outer connected component becomes surface 2, the inner becomes surface 1. Requires a complete, watertight mesh. |

#### Stage 2: Geometric matching

| Parameter | Default | Description |
|-----------|---------|-------------|
| `max_distance_nm` | `8.0` | Hard cap (nm) on the source–target distance. Pairs farther apart are never considered. This value is also reused in step 3 as the maximum allowed inflection-point thickness — set it to the largest physically plausible membrane thickness for your sample. |
| `max_angle` | `10.0` | Half-angle (degrees) of the normal-aligned search cone. Candidate targets whose lateral deviation from the source normal exceeds this angle are rejected. Larger values allow more oblique matches (useful for curved membranes); smaller values enforce stricter collinearity. |
| `direction` | `"1to2"` | Which surface seeds the ray casting. `"1to2"` iterates over surface-1 points and finds their nearest surface-2 partner inside the cone; `"2to1"` reverses the roles. The number of matched pairs and their distribution can differ slightly between directions. |
| `use_gpu` | `False` | Use the CUDA kernel when a compatible GPU is available. Automatically falls back to the CPU KDTree path if CUDA is not available. |
| `pixel_size_nm` | `None` | Voxel size in nm. Overrides the value read from the MRC header. Required when the segmentation was saved without voxel-size metadata (header reports 0 Å). |

#### Stage 3: Intensity profiles

| Parameter | Default | Description |
|-----------|---------|-------------|
| `profile_half_width_nm` | `5.5` | Half-width of each sampled 1-D intensity profile on both sides of the matched-pair midpoint (nm). The full profile spans `2 × profile_half_width_nm`. Should be large enough to capture both outward maxima beyond the leaflet minima, but not so large that unrelated density leaks into the profile. Typical values: `5.0`–`7.0` nm. |
| `max_distance_nm` | `8.0` | Rows whose inflection-point thickness (or minima-to-minima distance for `minima_only` rows) exceeds this value are excluded from the exported `*_thickness.csv`. Should match the geometric matching cap. |
| `analyzer` | - | `IntensityProfileAnalyzer()` detection parameter object (see table above). |

---

### `IntensityProfileAnalyzer` — settings for analyzing the extracted intensity profiles

The `IntensityProfileAnalyzer` object holds all parameters for step 3 (intensity profile-based search for minima, outward maxima, and profile inflection points). Create it once and pass it to `run_full_pipeline` or `analyse_intensity_profiles`.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `smooth_sigma_intensity_profiles` | `0.5` | Gaussian smoothing applied along the profile axis before peak detection. Increase if profiles are noisy; decrease (or set to 0) to preserve fine structure. |
| `extrema_prominence_threshold` | `0.1` | Minimum prominence for minima and maxima found by `find_peaks`. Lower values detect shallower features; higher values suppress noise peaks. |
| `minima_search_nm` | `(3.0, 4.0)` | Search window applied symmetrically around each matched segmentation surface point when looking for the bilayer minima. The pipeline first tries the narrower window; if no valid minimum pair is found, it retries with the wider window. |
| `anchor_search_nm` | `4.0` | Half-width (nm) of the outward search window beyond each identified minimum, within which the pipeline searches for an outward intensity maximum. This maximum marks where the density returns to baseline outside the bilayer and anchors the inflection-point calculation. See below for what happens when no maximum is found. |
| `n_jobs` | `-1` | Worker count for parallel profile processing. `-1` = all available cores; `1` = serial. |

**What happens when an outward maximum is not found on one side.**  
If no maximum is detected within `anchor_search_nm` of a bilayer minimum, the pipeline borrows the good side's geometry: the distance from that side's minimum to its outward maximum (the "span") is mirrored to predict where the missing anchor should be. The inflection point is then computed on the predicted slope segment, exactly as it would be for a real maximum. Profiles resolved this way are labeled `max_anchor` or `anchor_max` in the `detection_mode` column, depending on which side was mirrored.  
If no outward maximum exists on *either* side, no inflection-point thickness is calculated; instead, only the distance between the two leaflet minima is reported (`minima_only` mode, `membrane_thickness_nm` = NaN).



In [4]:
membrane_labels = {
    "ER": 1,
    "IMM": 2,
    "OMM": 3,
}

surface_separation_mode = {
    "ER": "planar",
    "IMM": "closed",   # less clear whether to go with open or close, as there will be flipping of the surfaces on the self-folding cristae
    "OMM": "closed",   # closed is important to get a meaningful separation of the two leaflets
}

analyzer = memthick.IntensityProfileAnalyzer(
    smooth_sigma_intensity_profiles=0.5,  # empirical, avoid over-smoothing
    minima_search_nm=(3.0, 4.0),
    anchor_search_nm=4.0,
    n_jobs=-1,
)

---

## Run full pipeline

Runs all three steps (surface extraction → geometric matching → intensity profile analysis) in a single call. Use this for a fresh run from a new segmentation. To re-enter the pipeline at a specific step, see the individual step sections below.


In [5]:
results = memthick.run_full_pipeline(
    segmentation_map=segmentation_path,
    tomogram_map=tomo_path,
    output_path=output_path,
    membrane_labels=membrane_labels,
    # -- Step 1: surface extraction --
    smooth_sigma_segmentation=2.0,    # to reduce "step-artifacts" on the surface due to the discrete nature of the segmentation
    surface_separation_mode=surface_separation_mode,
    refine_normals=True,
    radius_hit=3.0,
    # -- Step 2: geometric point matching --
    max_distance_nm=8.0,
    max_angle=10.0,
    direction="1to2",
    use_gpu=False,                    # set to True if you have a GPU
    # -- Step 3: intensity profiles analysis --
    extract_intensity_profiles=True,
    profile_half_width_nm=5.5,
    analyzer=analyzer,
    # -- File saving options --
    save_split_surface_meshes=True,   # to visualize surface splitting (will produce a .ply mesh file)
    save_vertices_mrc=True,           # to visualize vertices positions (rounds vertices to the nearest integer)
    save_thickness_mrc=True,          # voxel values correspond to thickness (aggregated from vertices that match back to this voxel)
)

2026-05-31 20:30:11,149 - Starting full membrane thickness analysis pipeline for inputs/2140_z150to400_segmented.mrc


2026-05-31 20:30:11,151 - Will attempt finding intensity profile features using tomogram: inputs/2140_z150to400.mrc
2026-05-31 20:30:11,152 - Step 1: Processing membrane segmentation
2026-05-31 20:30:11,774 - 
Processing ER (label 1)
2026-05-31 20:30:11,863 - Extracting surface points (mesh vertices) with step size 1 from segmentation after smoothing with sigma 2.0...


Applying Gaussian smoothing (sigma=2.0)...
Smoothed field range: [0.000, 0.997]
Running marching cubes (level=0.5, step_size=1)...


2026-05-31 20:30:27,955 - Extracted 477614 mesh vertices
2026-05-31 20:30:52,397 - Separating surfaces (mode='planar')...
2026-05-31 20:30:52,424 - Successfully separated membrane surfaces:
2026-05-31 20:30:52,424 - Surface 1: 242035 points
2026-05-31 20:30:52,425 - Surface 2: 235579 points
2026-05-31 20:30:52,426 - Refining normals within surface 1...


Refining normals (radius=3.0, batch=2000, iter=1, mask=242035/477614)
Iteration 1/1


...

2026-05-31 20:39:13,127 - Saved split mesh: outputs/2140_z150to400_segmented_OMM_surface1.ply


Saved mesh to outputs/2140_z150to400_segmented_OMM_surface1.ply
  Vertices: 617,884
  Faces: 1,231,092


2026-05-31 20:39:22,495 - Saved split mesh: outputs/2140_z150to400_segmented_OMM_surface2.ply
2026-05-31 20:39:22,501 - 
Saving outputs for OMM:
2026-05-31 20:39:22,503 - Vertices: 1255650, Surface 1: 617884, Surface 2: 637766


Saved mesh to outputs/2140_z150to400_segmented_OMM_surface2.ply
  Vertices: 637,766
  Faces: 1,270,737


2026-05-31 20:39:24,187 - Saved MRC to outputs/2140_z150to400_segmented_OMM_vertices.mrc
2026-05-31 20:39:35,651 - Saved CSV with 1255650 vertices to outputs/2140_z150to400_segmented_OMM_vertices_normals.csv
2026-05-31 20:39:35,665 - 
Step 2: Matching points between surfaces
2026-05-31 20:39:35,666 - 
Processing surface matches for ER
2026-05-31 20:39:35,930 - Loading data from outputs/2140_z150to400_segmented_ER_vertices_normals.csv...
2026-05-31 20:39:36,438 - Starting surface matching (1to2)...
2026-05-31 20:39:36,439 - Using all available CPU threads (numba default)
2026-05-31 20:39:36,439 - Matching surface 1 points to surface 2...
2026-05-31 20:39:36,440 - Starting CPU surface matching with 477614 points...
2026-05-31 20:39:36,441 - Source points: 242035, target points: 235579, max match distance: 8.00 nm (10.20 vox)
2026-05-31 20:39:36,441 - Max angle: 10.0 degrees
2026-05-31 20:39:36,448 - Building KD-tree for target points...
2026-05-31 20:39:36,535 - Querying KD-tree in batch

---

## Run step 1: surface extraction

Extract surface meshes from each membrane label and write `*_vertices_normals.csv`. Run this step when you want to:

- Inspect the extracted surfaces before committing to a full pipeline run.
- Try different `surface_separation_mode` or `smooth_sigma_segmentation` values.
- Run steps 2 and 3 separately afterwards (pass the output CSVs to `match_points`).

**Output per membrane:** one CSV with columns `x_voxel`, `y_voxel`, `z_voxel` (float, sub-voxel marching-cubes coordinates), `normal_x/y/z`, `surface1`, `surface2`.

> `snap_vertices_to_boundary=False` is the recommended setting when you intend to extract intensity profiles in step 3. Sub-voxel vertex positions center the sampled line profiles more accurately between the two leaflets. Set it to `True` only if you are using the segmentation geometry as a direct measure of thickness (no tomogram intensity values).


In [6]:
# Returns {membrane_name: path_to_vertices_normals_csv}
vertices_csvs = memthick.process_membrane_segmentation(
    segmentation_map=segmentation_path,
    output_path=output_path,
    membrane_labels=membrane_labels,
    smooth_sigma_segmentation=2.0,
    surface_separation_mode=surface_separation_mode,
    refine_normals=True,
    radius_hit=3.0,
    snap_vertices_to_boundary=False, # set to False if you are extracting intensity profiles from the tomogram; set to True if you are using the segmentation as a pure measure of thickness
    save_split_surface_meshes=False, # to visualize surface splitting (will produce a .ply mesh file)
    save_vertices_mrc=False, # to visualize vertices positions (rounds vertices to the nearest integer)
)
print(vertices_csvs)

2026-05-31 21:00:42,041 - 
Processing ER (label 1)
2026-05-31 21:00:42,171 - Extracting surface points (mesh vertices) with step size 1 from segmentation after smoothing with sigma 2.0...


Applying Gaussian smoothing (sigma=2.0)...
Smoothed field range: [0.000, 0.997]
Running marching cubes (level=0.5, step_size=1)...


2026-05-31 21:01:03,176 - Extracted 477614 mesh vertices
2026-05-31 21:01:45,139 - Separating surfaces (mode='planar')...
2026-05-31 21:01:45,209 - Successfully separated membrane surfaces:
2026-05-31 21:01:45,211 - Surface 1: 242035 points
2026-05-31 21:01:45,212 - Surface 2: 235579 points
2026-05-31 21:01:45,212 - Refining normals within surface 1...


Refining normals (radius=3.0, batch=2000, iter=1, mask=242035/477614)
Iteration 1/1


100%|██████████| 122/122 [00:18<00:00,  6.70it/s]
2026-05-31 21:02:03,778 - Refining normals within surface 2...


...

Normal refinement complete.
Refining normals (radius=3.0, batch=2000, iter=1, mask=637766/1255650)
Iteration 1/1


100%|██████████| 319/319 [00:56<00:00,  5.67it/s]
2026-05-31 21:11:33,581 - 
Saving outputs for OMM:
2026-05-31 21:11:33,584 - Vertices: 1255650, Surface 1: 617884, Surface 2: 637766


Normal refinement complete.


2026-05-31 21:11:45,151 - Saved CSV with 1255650 vertices to outputs/2140_z150to400_segmented_OMM_vertices_normals.csv


{'ER': 'outputs/2140_z150to400_segmented_ER_vertices_normals.csv', 'IMM': 'outputs/2140_z150to400_segmented_IMM_vertices_normals.csv', 'OMM': 'outputs/2140_z150to400_segmented_OMM_vertices_normals.csv'}


---

## Run step 2: surface point matching

Use this when you want to re-run geometric matching with different parameters — e.g. a wider distance cap or a different cone angle — without repeating the meshing step.

**Input:** `*_vertices_normals.csv` from step 1.  
**Output per membrane:** `*_matched_points.csv` + `*_matched_points_stats.txt`.

The matched-points CSV contains one row per successfully paired point on surface 1, with the coordinates of both the surface-1 and surface-2 points (`x1_voxel … z2_voxel`) and the Euclidean distance between them (`match_distance_nm`).  
This distance is a **geometric** quantity derived purely from the segmentation — it is the distance along the normal between the two surface meshes.

> **`match_distance_nm` vs `membrane_thickness_nm`.**  
> `match_distance_nm` (step 2 output) is the distance between the two geometrically matched segmentation surface points — it describes the segmentation, not the underlying biology.  
> `membrane_thickness_nm` (step 3 output) is the distance between the inflection points of the tomogram intensity profiles and is measured from the tomographic densities.


In [7]:
# Point to the CSV(s) produced by step 1
vertices_csvs = {
    "ER": f"{output_path}/2140_z150to400_segmented_ER_vertices_normals.csv",
    "IMM": f"{output_path}/2140_z150to400_segmented_IMM_vertices_normals.csv",
    "OMM": f"{output_path}/2140_z150to400_segmented_OMM_vertices_normals.csv",
}

for membrane_name, csv in vertices_csvs.items():
    matched_csv, stats_file = memthick.match_points(
        segmentation_map=segmentation_path,
        input_csv=csv,
        output_path=output_path,
        max_distance_nm=8.0,
        max_angle=10.0,
        direction="1to2",
        use_gpu=False,
        surface_separation_mode=surface_separation_mode[membrane_name], # not actively used but written in the log file
    )
    print(f"{membrane_name}: {matched_csv}")

2026-05-31 21:12:02,636 - Loading data from outputs/2140_z150to400_segmented_ER_vertices_normals.csv...
2026-05-31 21:12:03,167 - Starting surface matching (1to2)...
2026-05-31 21:12:03,167 - Using all available CPU threads (numba default)
2026-05-31 21:12:03,168 - Matching surface 1 points to surface 2...
2026-05-31 21:12:03,168 - Starting CPU surface matching with 477614 points...
2026-05-31 21:12:03,170 - Source points: 242035, target points: 235579, max match distance: 8.00 nm (10.20 vox)
2026-05-31 21:12:03,170 - Max angle: 10.0 degrees
2026-05-31 21:12:03,178 - Building KD-tree for target points...
2026-05-31 21:12:03,267 - Querying KD-tree in batches of 200000 (242035 source points)...
2026-05-31 21:12:25,222 - KD-tree neighborhood search completed in 21.96 seconds
2026-05-31 21:12:25,225 - Found 1990999 potential matches across all source points
2026-05-31 21:12:25,226 - Resolving one-to-one surface matches...
2026-05-31 21:12:26,988 - Resolved 211259 one-to-one surface matches

ER: outputs/2140_z150to400_segmented_ER_vertices_normals_matched_points.csv


2026-05-31 21:12:30,154 - Loading data from outputs/2140_z150to400_segmented_IMM_vertices_normals.csv...
2026-05-31 21:12:32,084 - Starting surface matching (1to2)...
2026-05-31 21:12:32,085 - Using all available CPU threads (numba default)
2026-05-31 21:12:32,085 - Matching surface 1 points to surface 2...
2026-05-31 21:12:32,086 - Starting CPU surface matching with 1857969 points...
2026-05-31 21:12:32,088 - Source points: 903196, target points: 954773, max match distance: 8.00 nm (10.20 vox)
2026-05-31 21:12:32,088 - Max angle: 10.0 degrees
2026-05-31 21:12:32,115 - Building KD-tree for target points...
2026-05-31 21:12:32,551 - Querying KD-tree in batches of 200000 (903196 source points)...
2026-05-31 21:14:37,062 - KD-tree neighborhood search completed in 124.51 seconds
2026-05-31 21:14:37,066 - Found 5490559 potential matches across all source points
2026-05-31 21:14:37,067 - Resolving one-to-one surface matches...
2026-05-31 21:14:44,224 - Resolved 809465 one-to-one surface matc

IMM: outputs/2140_z150to400_segmented_IMM_vertices_normals_matched_points.csv


2026-05-31 21:14:56,378 - Loading data from outputs/2140_z150to400_segmented_OMM_vertices_normals.csv...
2026-05-31 21:14:57,703 - Starting surface matching (1to2)...
2026-05-31 21:14:57,703 - Using all available CPU threads (numba default)
2026-05-31 21:14:57,704 - Matching surface 1 points to surface 2...
2026-05-31 21:14:57,704 - Starting CPU surface matching with 1255650 points...
2026-05-31 21:14:57,706 - Source points: 617884, target points: 637766, max match distance: 8.00 nm (10.20 vox)
2026-05-31 21:14:57,706 - Max angle: 10.0 degrees
2026-05-31 21:14:57,727 - Building KD-tree for target points...
2026-05-31 21:14:57,989 - Querying KD-tree in batches of 200000 (617884 source points)...
2026-05-31 21:16:17,600 - KD-tree neighborhood search completed in 79.61 seconds
2026-05-31 21:16:17,604 - Found 3478677 potential matches across all source points
2026-05-31 21:16:17,604 - Resolving one-to-one surface matches...
2026-05-31 21:16:21,579 - Resolved 580064 one-to-one surface match

OMM: outputs/2140_z150to400_segmented_OMM_vertices_normals_matched_points.csv


### (Optional) per-surface MRC volumes from segmentation surface point matching

Rasterizes the matched-point distances (purely based on segmentation surfaces) onto two per-surface MRC volumes and CSVs — one for each bilayer surface. Use this when you want a spatial map of `match_distance_nm` (segmentation thickness) to visually overlay on the tomogram.

Each MRC voxel receives the **median** `match_distance_nm` (segmentation thickness)across all vertices that round to that integer position. Voxels with no matching vertex are zero.

**Key parameters for `save_thickness_mrc`:**

| Parameter | Description |
|-----------|-------------|
| `thickness_col` | Column to rasterize. Use `"match_distance_nm"` for step-2 geometry output. |
| `coord_suffix` | Coordinate column suffix. Use `"voxel"` to read `x1_voxel / x2_voxel` (step-2 CSV format). |


In [8]:
# Aggreagates vertices that match back to the same voxel into a single voxel value
matched_csvs = {
    "ER": f"{output_path}/2140_z150to400_segmented_ER_matched_points.csv",
    "IMM": f"{output_path}/2140_z150to400_segmented_IMM_matched_points.csv",
    "OMM": f"{output_path}/2140_z150to400_segmented_OMM_matched_points.csv",
}

for membrane_name, csv in matched_csvs.items():
    saved = memthick.save_thickness_mrc(
        thickness_csv=csv,
        segmentation_map=segmentation_path,
        output_path=output_path,
        thickness_col="match_distance_nm",
        coord_suffix="voxel",
    )
    print(f"{membrane_name}: {saved}")

Discretizing 211259 rows to integer voxels (0 non-finite dropped)
Leaflet 1: 120317 unique voxels; Leaflet 2: 115755 unique voxels
Saved Surface 1 volume: outputs/2140_z150to400_segmented_ER_surface1_match_distance_discretized_nm.mrc
Saved Surface 1 discretized CSV: outputs/2140_z150to400_segmented_ER_surface1_match_distance_discretized_nm.csv
Saved Surface 2 volume: outputs/2140_z150to400_segmented_ER_surface2_match_distance_discretized_nm.mrc
Saved Surface 2 discretized CSV: outputs/2140_z150to400_segmented_ER_surface2_match_distance_discretized_nm.csv
ER: {'surface1_mrc': PosixPath('outputs/2140_z150to400_segmented_ER_surface1_match_distance_discretized_nm.mrc'), 'surface1_csv': PosixPath('outputs/2140_z150to400_segmented_ER_surface1_match_distance_discretized_nm.csv'), 'surface2_mrc': PosixPath('outputs/2140_z150to400_segmented_ER_surface2_match_distance_discretized_nm.mrc'), 'surface2_csv': PosixPath('outputs/2140_z150to400_segmented_ER_surface2_match_distance_discretized_nm.csv')

---

## Run step 3: intensity profile analysis

Use this when you want to iterate on `IntensityProfileAnalyzer` settings — e.g. adjusting profile smoothing, minima search windows, or the outward maximum search range — without re-running meshing or geometric matching.

**Input:** `*_matched_points.csv` from step 2 + the original tomogram MRC.  
**Output per membrane:** `*_thickness.csv`, `*_int_profiles.pkl`, `*_boundary_stats.txt`.

For each matched pair, the pipeline:

1. Extracts a short 1-D intensity profile from the tomogram along the vector connecting the two matched points, extended by `profile_half_width_nm` on each side of the midpoint.
2. Detects the two **bilayer minima** (phosphate headgroup regions) and the **central maximum** (hydrophobic core) within search windows anchored on the matched segmentation surface points.
3. Searches outward from each minimum for a **maximum** (close to where the intensity returns to baseline). The inflection point on the steepest slope between each minimum and its outward maximum defines the membrane boundary.
4. Applies a final `max_distance_nm` cap: rows whose measured inflection-point thickness exceeds this value are excluded from the exported CSV.

See [`memthick_analysis.ipynb`](memthick_analysis.ipynb) for a detailed description of the output columns, detection modes (`max_max`, `max_anchor`, `minima_only`), and how to interpret the intensity profile plots.


In [ ]:
# Point to the CSV(s) produced by step 2
matched_csvs = {
    "ER": f"{output_path}/2140_z150to400_segmented_ER_matched_points.csv",
    "IMM": f"{output_path}/2140_z150to400_segmented_IMM_matched_points.csv",
    "OMM": f"{output_path}/2140_z150to400_segmented_OMM_matched_points.csv",
}

for membrane_name, csv in matched_csvs.items():
    intensity_results = memthick.analyse_intensity_profiles(
        thickness_csv=csv,
        tomogram_map=tomo_path,
        segmentation_path=segmentation_path,
        membrane_label=membrane_name,
        output_path=output_path,
        profile_half_width_nm=5.5,
        max_distance_nm=8.0,
        analyzer=analyzer,
    )
    print(f"{membrane_name}: {intensity_results.get('saved_files', {})}")

2026-05-31 21:24:01,937 - Writing pipeline diagnostics to 2140_z150to400_segmented_ER_2140_z150to400_analysis.log (under /Users/desislavaglushkova/code/local_tests/cryocat_refactor/example_notebooks/memthick/outputs)
2026-05-31 21:24:02,264 - === Extracting intensity profiles ===
2026-05-31 21:24:02,265 - Tomogram MRC: 2140_z150to400.mrc
2026-05-31 21:24:02,266 -           path: /Users/desislavaglushkova/code/local_tests/cryocat_refactor/example_notebooks/memthick/inputs/2140_z150to400.mrc
2026-05-31 21:24:02,267 - Segmentation MRC: 2140_z150to400_segmented.mrc
2026-05-31 21:24:02,268 -             path: /Users/desislavaglushkova/code/local_tests/cryocat_refactor/example_notebooks/memthick/inputs/2140_z150to400_segmented.mrc
2026-05-31 21:24:02,268 - Matched-point CSV: 2140_z150to400_segmented_ER_matched_points.csv
2026-05-31 21:24:02,269 -              path: /Users/desislavaglushkova/code/local_tests/cryocat_refactor/example_notebooks/memthick/outputs/2140_z150to400_segmented_ER_match

ER: {'thickness_csv': PosixPath('outputs/2140_z150to400_segmented_ER_thickness.csv'), 'int_profiles': PosixPath('outputs/2140_z150to400_segmented_ER_int_profiles.pkl'), 'statistics': PosixPath('outputs/2140_z150to400_segmented_ER_boundary_stats.txt')}


2026-05-31 21:27:14,177 - === Extracting intensity profiles ===
2026-05-31 21:27:14,177 - Tomogram MRC: 2140_z150to400.mrc
2026-05-31 21:27:14,179 -           path: /Users/desislavaglushkova/code/local_tests/cryocat_refactor/example_notebooks/memthick/inputs/2140_z150to400.mrc
2026-05-31 21:27:14,179 - Segmentation MRC: 2140_z150to400_segmented.mrc
2026-05-31 21:27:14,180 -             path: /Users/desislavaglushkova/code/local_tests/cryocat_refactor/example_notebooks/memthick/inputs/2140_z150to400_segmented.mrc
2026-05-31 21:27:14,181 - Matched-point CSV: 2140_z150to400_segmented_IMM_matched_points.csv
2026-05-31 21:27:14,182 -              path: /Users/desislavaglushkova/code/local_tests/cryocat_refactor/example_notebooks/memthick/outputs/2140_z150to400_segmented_IMM_matched_points.csv
2026-05-31 21:27:14,182 -              rows: 809,465; usable coordinate pairs (non-NaN): 809,465
2026-05-31 21:27:14,291 - Z-score normalizing tomogram before profile sampling...
2026-05-31 21:27:44,92

### (Optional) per-surface MRC volumes from full thickness analysis

Rasterizes inflection-point-based membrane thickness onto two per-surface MRC volumes and CSVs. Run this after `analyse_intensity_profiles` has written a `*_thickness.csv`.

Compared to the step-2 geometry MRC above, this uses `membrane_thickness_nm` (inflection-point distances from the tomogram) rather than `match_distance_nm` (segmentation-surface distances). The coordinate columns also differ: the step-3 CSV uses corrected coordinates (`x1_corr_voxel`, `x2_corr_voxel`, …) that reflect the inflection-point positions rather than the original matched surface points.

**Key parameters for `save_thickness_mrc`:**

| Parameter | Default | Description |
|-----------|---------|-------------|
| `thickness_col` | `"membrane_thickness_nm"` | Column to rasterize. `minima_only` rows have `NaN` here and are automatically skipped. |
| `coord_suffix` | `"corr_voxel"` | Coordinate column suffix. Matches `x1_corr_voxel / x2_corr_voxel` in `*_thickness.csv`. |


In [11]:
thickness_csvs = {
    "ER": f"{output_path}/2140_z150to400_segmented_ER_thickness.csv",
    "IMM": f"{output_path}/2140_z150to400_segmented_IMM_thickness.csv",
    "OMM": f"{output_path}/2140_z150to400_segmented_OMM_thickness.csv",
}

for membrane_name, csv in thickness_csvs.items():
    saved = memthick.save_thickness_mrc(
        thickness_csv=csv,
        segmentation_map=segmentation_path,
        output_path=output_path,
        coord_suffix="corr_voxel",              # default, matches *_thickness.csv
        thickness_col="membrane_thickness_nm"  # default
    )
    print(f"{membrane_name}: {saved}")

Discretizing 148714 rows to integer voxels (23392 non-finite dropped)
Leaflet 1: 107913 unique voxels; Leaflet 2: 106651 unique voxels
Saved Surface 1 volume: outputs/2140_z150to400_segmented_ER_surface1_membrane_thickness_discretized_nm.mrc
Saved Surface 1 discretized CSV: outputs/2140_z150to400_segmented_ER_surface1_membrane_thickness_discretized_nm.csv
Saved Surface 2 volume: outputs/2140_z150to400_segmented_ER_surface2_membrane_thickness_discretized_nm.mrc
Saved Surface 2 discretized CSV: outputs/2140_z150to400_segmented_ER_surface2_membrane_thickness_discretized_nm.csv
ER: {'surface1_mrc': PosixPath('outputs/2140_z150to400_segmented_ER_surface1_membrane_thickness_discretized_nm.mrc'), 'surface1_csv': PosixPath('outputs/2140_z150to400_segmented_ER_surface1_membrane_thickness_discretized_nm.csv'), 'surface2_mrc': PosixPath('outputs/2140_z150to400_segmented_ER_surface2_membrane_thickness_discretized_nm.mrc'), 'surface2_csv': PosixPath('outputs/2140_z150to400_segmented_ER_surface2_mem